## GraphDB KC 데이터셋

### 1. 필요한 라이브러리

In [1]:
import pandas as pd
from neo4j import GraphDatabase
import json

### 2. Neo4j 연결

In [ ]:
NEO4J_URI=""
NEO4J_USERNAME=""
NEO4J_PASSWORD=""
NEO4J_DATABASE=""
AURA_INSTANCEID=""
AURA_INSTANCENAME=""

In [3]:
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

def close_driver():
    driver.close()

print(driver.get_server_info())

### 3. 데이터 처리


In [4]:
NODE_DATA = '../data/wikipedia/wikipedia_node.csv'
EDGE_DATA = '../data/wikipedia/wikipedia_edge.csv'

nodes_df = pd.read_csv(NODE_DATA)
edges_df = pd.read_csv(EDGE_DATA)

print(f"노드 수: {len(nodes_df)}")
print(f"엣지 수: {len(edges_df)}")

# 데이터 확인
print("\n=== 노드 샘플 ===")
print(nodes_df.head())

print("\n=== 엣지 샘플 ===")
print(edges_df.head())

노드 수: 5934
엣지 수: 25166

=== 노드 샘플 ===
                                        name  \
0  (1+ε)-approximate nearest neighbor search   
1                                        .ai   
2                              Loss function   
3                                      01.AI   
4              1.58-bit large language model   

                                                link  \
0  https://en.wikipedia.org/wiki/(1+ε)-approximat...   
1                  https://en.wikipedia.org/wiki/.ai   
2    https://en.wikipedia.org/wiki/0-1_loss_function   
3                https://en.wikipedia.org/wiki/01.AI   
4  https://en.wikipedia.org/wiki/1.58-bit_large_l...   

                                               alias  \
0  (1+epsilon)-approximate nearest neighbor searc...   
1  .ai, ai, dot ai, dot-ai, dot ai domain, ai domain   
2  0-1 loss, zero-one loss, 01 loss function, 0 1...   
3         01ai, 01 ai, 01-AI, 01.ai, 01ai.com, 01 AI   
4  1.58 bit large language model, 1.58-bit LLM, 1...   


In [5]:
import ast

# categories를 문자열에서 리스트로 변환
def parse_categories(cat_str):
    try:
        return ast.literal_eval(cat_str)
    except:
        return []

nodes_df['categories'] = nodes_df['categories'].apply(parse_categories)

# alias 처리 (필요한 경우)
nodes_df['alias'] = nodes_df['alias'].fillna('')

print("데이터 전처리 완료")
print(nodes_df.head())

데이터 전처리 완료
                                        name  \
0  (1+ε)-approximate nearest neighbor search   
1                                        .ai   
2                              Loss function   
3                                      01.AI   
4              1.58-bit large language model   

                                                link  \
0  https://en.wikipedia.org/wiki/(1+ε)-approximat...   
1                  https://en.wikipedia.org/wiki/.ai   
2    https://en.wikipedia.org/wiki/0-1_loss_function   
3                https://en.wikipedia.org/wiki/01.AI   
4  https://en.wikipedia.org/wiki/1.58-bit_large_l...   

                                               alias  \
0  (1+epsilon)-approximate nearest neighbor searc...   
1  .ai, ai, dot ai, dot-ai, dot ai domain, ai domain   
2  0-1 loss, zero-one loss, 01 loss function, 0 1...   
3         01ai, 01 ai, 01-AI, 01.ai, 01ai.com, 01 AI   
4  1.58 bit large language model, 1.58-bit LLM, 1...   

                          

### 4. Node 생성

In [ ]:
def create_constraints(tx):
    # CONSTRAINT
    try:
        tx.run("CREATE CONSTRAINT keyword_name_unique IF NOT EXISTS FOR (k:Keyword) REQUIRE k.name IS UNIQUE")
        print("Uniqueness constraint 생성 완료")
    except Exception as e:
        print(f"Constraint 생성 중 오류 (이미 존재할 수 있음): {e}")

def create_indexes(tx):
    # INDEX
    try:
        tx.run("CREATE INDEX keyword_name_index IF NOT EXISTS FOR (k:Keyword) ON (k.name)")
        print("Index 생성 완료")
    except Exception as e:
        print(f"Index 생성 중 오류: {e}")

with driver.session() as session:
    session.execute_write(create_constraints)
    session.execute_write(create_indexes)

Uniqueness constraint 생성 완료
Index 생성 완료


In [7]:
def create_keyword_node(tx, name, link, alias, categories):
    query = """
    MERGE (k:Keyword {name: node.name})
    SET k.link = node.link,
        k.alias = node.alias,
        k.categories = node.categories
    """
    result = tx.run(query, name=name, link=link, alias=alias, categories=categories)
    return result.single()['node_id']

def create_nodes_batch(tx, nodes_batch):
    query = """
    UNWIND $nodes as node
    MERGE (k:Keyword {name: node.name})
    SET k.link = node.link,
        k.alias = node.alias,
        k.categories = node.categories
    """
    tx.run(query, nodes=nodes_batch)

In [9]:
batch_size = 1000
total_nodes = len(nodes_df)

with driver.session() as session:
    for i in range(0, total_nodes, batch_size):
        batch = nodes_df.iloc[i:i+batch_size]
        nodes_batch = [
            {
                'name': row['name'],
                'link': row['link'],
                'alias': row['alias'],
                'categories': row['categories']
            }
            for _, row in batch.iterrows()
        ]
        session.execute_write(create_nodes_batch, nodes_batch)
        print(f"노드 생성 진행: {min(i+batch_size, total_nodes)}/{total_nodes}")

print(f"\n총 생성 노드: {total_nodes}")

[#9CE2]  _: <CONNECTION> error: Failed to write data to connection ResolvedIPv4Address(('34.126.114.186', 7687)) (ResolvedIPv4Address(('34.126.114.186', 7687))): ConnectionResetError(104, 'Connection reset by peer')
Unable to retrieve routing information
Transaction failed and will be retried in 1.1765610012017191s (Unable to retrieve routing information)
[#9CE6]  _: <CONNECTION> error: Failed to write data to connection IPv4Address(('si-7083bdc2-3e08.production-orch-0066.neo4j.io', 7687)) (ResolvedIPv4Address(('34.126.114.186', 7687))): ConnectionResetError(104, 'Connection reset by peer')
Transaction failed and will be retried in 2.0149405285433537s (Failed to write data to connection IPv4Address(('si-7083bdc2-3e08.production-orch-0066.neo4j.io', 7687)) (ResolvedIPv4Address(('34.126.114.186', 7687))))


노드 생성 진행: 1000/5934
노드 생성 진행: 2000/5934
노드 생성 진행: 3000/5934
노드 생성 진행: 4000/5934
노드 생성 진행: 5000/5934
노드 생성 진행: 5934/5934

총 생성 노드: 5934


### 5. Edge 생성

In [ ]:
def create_prereq_relationship(tx, source_name, target_name, strength, reason=None):
    query = """
    MATCH (source:Keyword {name: edge.source_name})
    MATCH (target:Keyword {name: edge.target_name})
    MERGE (source)-[r:PREREQ]->(target)
    SET r.strength = edge.strength,
        r.reason = edge.reason
    """
    result = tx.run(query, 
                    source_name=source_name, 
                    target_name=target_name, 
                    strength=strength, 
                    reason=reason)
    return result.single()

def create_edges_batch(tx, edges_batch):
    query = """
    UNWIND $edges as edge
    MATCH (source:Keyword {name: edge.source_name})
    MATCH (target:Keyword {name: edge.target_name})
    MERGE (source)-[r:PREREQ]->(target)
    SET r.strength = edge.strength,
        r.reason = edge.reason
    """
    tx.run(query, edges=edges_batch)


In [11]:
batch_size = 1000
total_edges = len(edges_df)

with driver.session() as session:
    for i in range(0, total_edges, batch_size):
        batch = edges_df.iloc[i:i+batch_size]
        edges_batch = [
            {
                'source_name': row['prereq_name'],
                'target_name': row['name'],
                'strength': float(row['strength']),
                'reason': row['reason'] if pd.notna(row['reason']) else None
            }
            for _, row in batch.iterrows()
        ]
        session.execute_write(create_edges_batch, edges_batch)
        print(f"엣지 생성 진행: {min(i+batch_size, total_edges)}/{total_edges}")

print(f"\n총 생성 엣지: {total_edges}")

[#A136]  _: <CONNECTION> error: Failed to write data to connection IPv4Address(('si-7083bdc2-3e08.production-orch-0066.neo4j.io', 7687)) (ResolvedIPv4Address(('34.126.114.186', 7687))): ConnectionResetError(104, 'Connection reset by peer')
Transaction failed and will be retried in 0.9638393983699063s (Failed to write data to connection IPv4Address(('si-7083bdc2-3e08.production-orch-0066.neo4j.io', 7687)) (ResolvedIPv4Address(('34.126.114.186', 7687))))


엣지 생성 진행: 1000/25166
엣지 생성 진행: 2000/25166
엣지 생성 진행: 3000/25166
엣지 생성 진행: 4000/25166
엣지 생성 진행: 5000/25166
엣지 생성 진행: 6000/25166
엣지 생성 진행: 7000/25166
엣지 생성 진행: 8000/25166
엣지 생성 진행: 9000/25166
엣지 생성 진행: 10000/25166
엣지 생성 진행: 11000/25166
엣지 생성 진행: 12000/25166
엣지 생성 진행: 13000/25166
엣지 생성 진행: 14000/25166
엣지 생성 진행: 15000/25166
엣지 생성 진행: 16000/25166
엣지 생성 진행: 17000/25166
엣지 생성 진행: 18000/25166
엣지 생성 진행: 19000/25166
엣지 생성 진행: 20000/25166
엣지 생성 진행: 21000/25166
엣지 생성 진행: 22000/25166
엣지 생성 진행: 23000/25166
엣지 생성 진행: 24000/25166
엣지 생성 진행: 25000/25166
엣지 생성 진행: 25166/25166

총 생성 엣지: 25166


### 6. 결과 확인 코드

In [13]:
def get_graph_stats(tx):
    # 노드 수
    node_count = tx.run("MATCH (k:Keyword) RETURN count(k) as count").single()['count']
    
    # 엣지 수
    edge_count = tx.run("MATCH ()-[r:PREREQ]->() RETURN count(r) as count").single()['count']
    
    return node_count, edge_count

def get_sample_nodes(tx, limit=5):
    result = tx.run("""
        MATCH (k:Keyword)
        RETURN k.name as name, k.link as link, k.categories as categories
        LIMIT $limit
    """, limit=limit)
    return [record.data() for record in result]

def get_sample_relationships(tx, limit=5):
    result = tx.run("""
        MATCH (source:Keyword)-[r:PREREQ]->(target:Keyword)
        RETURN source.name as source, target.name as target, 
               r.strength as strength, r.reason as reason
        LIMIT $limit
    """, limit=limit)
    return [record.data() for record in result]

# 통계 확인
with driver.session() as session:
    node_count, edge_count = session.execute_read(get_graph_stats)
    print(f"=== 그래프 통계 ===")
    print(f"총 노드 수: {node_count}")
    print(f"총 엣지 수: {edge_count}")
    
    print(f"\n=== 샘플 노드 (5개) ===")
    sample_nodes = session.execute_read(get_sample_nodes)
    for node in sample_nodes:
        print(f"- {node['name']}: {node['link']}")
        print(f"  Categories: {node['categories']}")
    
    print(f"\n=== 샘플 관계 (5개) ===")
    sample_rels = session.execute_read(get_sample_relationships)
    for rel in sample_rels:
        print(f"- {rel['source']} -[PREREQ]-> {rel['target']}")
        print(f"  Strength: {rel['strength']}, Reason: {rel['reason']}")


[#A6E6]  _: <CONNECTION> error: Failed to write data to connection ResolvedIPv4Address(('34.126.114.186', 7687)) (ResolvedIPv4Address(('34.126.114.186', 7687))): ConnectionResetError(104, 'Connection reset by peer')
Unable to retrieve routing information
Transaction failed and will be retried in 0.8164932602738462s (Unable to retrieve routing information)
[#A6A6]  _: <CONNECTION> error: Failed to write data to connection IPv4Address(('si-7083bdc2-3e08.production-orch-0066.neo4j.io', 7687)) (ResolvedIPv4Address(('34.126.114.186', 7687))): ConnectionResetError(104, 'Connection reset by peer')
Transaction failed and will be retried in 1.824864702557879s (Failed to write data to connection IPv4Address(('si-7083bdc2-3e08.production-orch-0066.neo4j.io', 7687)) (ResolvedIPv4Address(('34.126.114.186', 7687))))
[#A9DE]  _: <CONNECTION> error: Failed to read from defunct connection ResolvedIPv4Address(('34.126.114.186', 7687)) (ResolvedIPv4Address(('34.126.114.186', 7687))): ConnectionResetError

=== 그래프 통계 ===
총 노드 수: 5934
총 엣지 수: 25167

=== 샘플 노드 (5개) ===
- (1+ε)-approximate nearest neighbor search: https://en.wikipedia.org/wiki/(1+ε)-approximate_nearest_neighbor_search
  Categories: ['Algorithms and data structures stubs', 'Approximation algorithms', 'Classification algorithms', 'Search algorithms']
- .ai: https://en.wikipedia.org/wiki/.ai
  Categories: ['Artificial intelligence', 'Communications in Anguilla', 'Country code top-level domains', 'Domain Name System Security Extensions']
- Loss function: https://en.wikipedia.org/wiki/0-1_loss_function
  Categories: []
- 01.AI: https://en.wikipedia.org/wiki/01.AI
  Categories: ['2023 establishments in China', '2023 in artificial intelligence', 'Artificial intelligence companies']
- 1.58-bit large language model: https://en.wikipedia.org/wiki/1.58-bit_large_language_model
  Categories: ['Large language models']

=== 샘플 관계 (5개) ===
- Algorithm -[PREREQ]-> (1+ε)-approximate nearest neighbor search
  Strength: 0.7, Reason: Understan